# Lab 3: Fine-Tuning for Radiology Report Summarization
## CCI Session 4

**Duration:** 15 minutes  
**GPU Required:** Enable T4 GPU in Runtime → Change runtime type  

### Clinical Scenario
> KHCC radiologists write detailed findings for every chest X-ray, CT, and MRI. The impression section summarizes the key takeaways. Automating impression generation could save hours of reporting time. You'll fine-tune a language model on real radiology reports to generate impressions from findings.

### Objective
- Load and explore a radiology report dataset
- Establish a baseline with an off-the-shelf model
- Fine-tune the model on radiology data
- Evaluate with ROUGE metrics and compare before/after

---
## Setup

First, install dependencies and verify GPU access.

In [ ]:
!pip install -q transformers datasets evaluate rouge_score accelerate

import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

if not torch.cuda.is_available():
    print("\n\u26a0\ufe0f WARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> Select T4 GPU")

---
## Cell 1: Load Radiology Reports

In [ ]:
# === CELL 1: LOAD RADIOLOGY REPORTS ===
from datasets import load_dataset

# TODO: Load the MIMIC-CXR summarization dataset
# ds = load_dataset("tgrex6/mimic-cxr-reports-summarization")

# TODO: Print the dataset structure and number of train/val examples
# print(ds)

# TODO: Look at 3 example reports — print findings and impression side by side
# Note the pattern: findings (detailed) -> impression (concise summary)
# for i in range(3):
#     print(f"--- Report {i+1} ---")
#     print(f"FINDINGS: {ds['train'][i]['findings'][:300]}...")
#     print(f"IMPRESSION: {ds['train'][i]['impression']}")
#     print()

# TODO: For training efficiency, take a subset of 1000 training examples
# small_ds = ds["train"].shuffle(seed=42).select(range(1000))
# print(f"Training subset size: {len(small_ds)}")

---
## Cell 2: Data Exploration

In [ ]:
# === CELL 2: DATA EXPLORATION ===

# TODO: What's the average length (words) of findings vs impressions?
# This tells us the compression ratio — how much summarization is needed
# findings_lengths = [len(x["findings"].split()) for x in small_ds]
# impression_lengths = [len(x["impression"].split()) for x in small_ds]
# print(f"Avg findings length: {sum(findings_lengths)/len(findings_lengths):.0f} words")
# print(f"Avg impression length: {sum(impression_lengths)/len(impression_lengths):.0f} words")
# print(f"Compression ratio: {sum(findings_lengths)/sum(impression_lengths):.1f}x")

# TODO: Check for any empty findings or impressions
# empty_findings = sum(1 for x in small_ds if not x["findings"].strip())
# empty_impressions = sum(1 for x in small_ds if not x["impression"].strip())
# print(f"Empty findings: {empty_findings}, Empty impressions: {empty_impressions}")

# TODO: Filter out empty rows if needed
# small_ds = small_ds.filter(lambda x: len(x["findings"].strip()) > 0 and len(x["impression"].strip()) > 0)
# print(f"Clean dataset size: {len(small_ds)}")

---
## Cell 3: Load Model & Tokenizer

In [ ]:
# === CELL 3: LOAD MODEL ===
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# TODO: Load the flan-t5-small model and tokenizer
# model_name = "google/flan-t5-small"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# TODO: Move model to GPU
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = model.to(device)
# print(f"Model loaded on: {device}")
# print(f"Model parameters: {model.num_parameters():,}")

---
## Cell 4: Baseline — Before Fine-Tuning

Let's see how the model performs **before** any training on radiology data.

In [ ]:
# === CELL 4: BASELINE — BEFORE FINE-TUNING ===
import evaluate

rouge = evaluate.load("rouge")

# We'll use these 5 reports for before/after comparison
test_samples = small_ds.select(range(5))

# TODO: Test the model on 5 radiology reports WITHOUT fine-tuning
# baseline_predictions = []
# for i, sample in enumerate(test_samples):
#     input_text = "Summarize the following radiology findings:\n" + sample["findings"]
#     inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
#     
#     with torch.no_grad():
#         outputs = model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True)
#     
#     prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     baseline_predictions.append(prediction)
#     
#     print(f"--- Report {i+1} ---")
#     print(f"ACTUAL:    {sample['impression'][:200]}")
#     print(f"PREDICTED: {prediction[:200]}")
#     print()

# TODO: Calculate ROUGE scores for baseline
# actual_impressions = [s["impression"] for s in test_samples]
# baseline_results = rouge.compute(predictions=baseline_predictions, references=actual_impressions)
# print("=== BASELINE ROUGE SCORES ===")
# for key, value in baseline_results.items():
#     print(f"{key}: {value:.4f}")

---
## Cell 5: Preprocessing

Tokenize the dataset so the model can learn from it.

In [ ]:
# === CELL 5: PREPROCESSING ===

# TODO: Define a preprocessing function that:
# 1. Adds a prefix: "Summarize the following radiology findings:\n"
# 2. Tokenizes inputs (findings) with max_length=512
# 3. Tokenizes targets (impressions) with max_length=128
# 4. Returns input_ids, attention_mask, labels

PREFIX = "Summarize the following radiology findings:\n"

def preprocess_function(examples):
    pass  # TODO: Implement this function
    # inputs = [PREFIX + finding for finding in examples["findings"]]
    # model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    # labels = tokenizer(examples["impression"], max_length=128, truncation=True, padding="max_length")
    # model_inputs["labels"] = labels["input_ids"]
    # return model_inputs

# TODO: Apply to the dataset
# tokenized_ds = small_ds.map(preprocess_function, batched=True, remove_columns=small_ds.column_names)
# print(f"Tokenized dataset: {tokenized_ds}")
# print(f"Features: {tokenized_ds.features}")

---
## Cell 6: Training Setup

In [ ]:
# === CELL 6: TRAINING SETUP ===
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# TODO: Define training arguments
# training_args = TrainingArguments(
#     output_dir="./radiology-summarizer",
#     num_train_epochs=3,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     learning_rate=5e-5,
#     weight_decay=0.01,
#     logging_steps=50,
#     save_strategy="epoch",
#     fp16=True,  # Mixed precision for T4 GPU speed
#     report_to="none",  # Disable wandb
# )

# TODO: Create data collator
# data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# TODO: Create Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_ds,
#     data_collator=data_collator,
# )
# print("Trainer ready!")

---
## Cell 7: Fine-Tune the Model

This will take approximately **5-10 minutes** on a T4 GPU with 1000 examples.

In [ ]:
# === CELL 7: FINE-TUNE THE MODEL ===

# TODO: Start training!
# trainer.train()
# Watch the training loss decrease over time

---
## Cell 8: After Fine-Tuning

Run the **same 5 reports** through the fine-tuned model and compare.

In [ ]:
# === CELL 8: AFTER FINE-TUNING ===

# TODO: Run the SAME 5 reports from Cell 4 through the fine-tuned model
# finetuned_predictions = []
# for i, sample in enumerate(test_samples):
#     input_text = PREFIX + sample["findings"]
#     inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
#     
#     with torch.no_grad():
#         outputs = model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True)
#     
#     prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     finetuned_predictions.append(prediction)
#     
#     print(f"--- Report {i+1} ---")
#     print(f"ACTUAL:         {sample['impression'][:200]}")
#     print(f"BEFORE TUNING:  {baseline_predictions[i][:200]}")
#     print(f"AFTER TUNING:   {prediction[:200]}")
#     print()

# TODO: Calculate ROUGE scores for fine-tuned model
# finetuned_results = rouge.compute(predictions=finetuned_predictions, references=actual_impressions)
# print("=== FINE-TUNED ROUGE SCORES ===")
# for key, value in finetuned_results.items():
#     print(f"{key}: {value:.4f}")

---
## Cell 9: Understanding ROUGE Metrics

Compare baseline vs fine-tuned performance across all ROUGE metrics.

In [ ]:
# === CELL 9: UNDERSTANDING ROUGE ===

# TODO: Print a side-by-side comparison table
# print(f"{'Metric':<12} {'Baseline':>10} {'Fine-Tuned':>12} {'Improvement':>12}")
# print("-" * 48)
# for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
#     base_val = baseline_results[metric]
#     ft_val = finetuned_results[metric]
#     improvement = ft_val - base_val
#     print(f"{metric:<12} {base_val:>10.4f} {ft_val:>12.4f} {improvement:>+12.4f}")

# Discussion questions:
# - ROUGE-1 measures unigram (single word) overlap
# - ROUGE-2 measures bigram (two-word phrase) overlap
# - ROUGE-L measures longest common subsequence
# Which metric improved the most? Why might that be?

---
## Stretch Challenge

Try modifying one hyperparameter and re-training to see if you can beat your results:
- Change `learning_rate` to `3e-5` or `1e-4`
- Change `num_train_epochs` to `5`
- Change `per_device_train_batch_size` to `4` or `16`

**Question:** Which change had the biggest impact on ROUGE scores?

In [ ]:
# === STRETCH CHALLENGE ===
# Try different hyperparameters here!
# Re-load the base model, change one setting, train, and compare


---
## KHCC Connection

This lab mirrors the radiology AI project being explored at KHCC where automated impression generation could reduce reporting time for the 200+ chest X-rays reviewed daily.

**Key takeaways:**
- Even a small model (flan-t5-small, 80M parameters) can learn radiology summarization patterns
- Fine-tuning on domain-specific data dramatically improves performance over a generic model
- ROUGE metrics provide a quantitative way to measure summarization quality
- In a real deployment, you would use a larger model, more data, and human evaluation alongside ROUGE

**Next steps at KHCC:**
- Scale to full dataset (93K reports) with a larger model
- Add Arabic language support for bilingual reports
- Integrate with the radiology information system (RIS) for real-time suggestions
- Conduct radiologist evaluation studies to validate clinical utility